# 09. A robotic arm: non-trivial connection topology

This notebook exercises the operator-overloaded syntax on a problem where the connections don't follow a clean series / parallel / loop pattern. The arm has:

- two **joint** actuators (shoulder and elbow), each with their own torque and power characteristics,
- a **controller** driving both joints and ingesting sensor data,
- a **sensor** measuring arm position,
- a **battery** sized by total mission energy.

The connection graph is genuinely non-trivial: the controller's power demand depends on both incoming sensor data and outgoing commands, both joints share the same battery via a power bus, and the shoulder torque has to support the elbow's own mass at the end of the shoulder arm. The Kleene iteration resolves the resulting cycles automatically.

The point of this notebook is that the resulting code reads as a paragraph of physical relationships, not a sequence of dict-juggling lambdas.


## Imports

In [1]:
from codesign import Module, Reals, System, solve

## Joint, sensor, controller, battery

Each subsystem is a small `Module`. The `Joint` class is reused for both shoulder and elbow with different `motor_density` parameters.


In [2]:
G = 9.81


class Joint(Module):
    F = {"torque": Reals(unit="N*m"), "speed": Reals(unit="rad/s")}
    R = {"mass":   Reals(unit="kg"),  "electric_power": Reals(unit="W")}

    def __init__(self, motor_density=0.20, efficiency=0.85):
        self.motor_density = motor_density
        self.efficiency = efficiency
        super().__init__()

    def h(self, f):
        return {
            "mass":           self.motor_density * f["torque"],
            "electric_power": f["torque"] * f["speed"] / self.efficiency,
        }


class Sensor(Module):
    F = {"sample_rate": Reals(unit="Hz")}
    R = {"power": Reals(unit="W"), "mass": Reals(unit="kg")}

    def h(self, f):
        return {"power": 0.02 * f["sample_rate"] + 0.5, "mass": 0.05}


class Controller(Module):
    F = {"input_rate": Reals(unit="Hz"), "command_rate": Reals(unit="Hz")}
    R = {"power":      Reals(unit="W"),  "mass":         Reals(unit="kg")}

    def h(self, f):
        return {
            "power": 0.05 * (f["input_rate"] + f["command_rate"]) + 2.0,
            "mass":  0.15,
        }


class Battery(Module):
    F = {"energy": Reals(unit="J")}
    R = {"mass":   Reals(unit="kg")}

    def __init__(self, specific_energy=1.8e6):
        self.specific_energy = specific_energy
        super().__init__()

    def h(self, f):
        return {"mass": f["energy"] / self.specific_energy}

## Wire everything together

Seven mission parameters drive the system; one resource (total mass) comes out. The eight constraint lines below capture the entire interconnection.


In [3]:
sys = System("robotic_arm")

# Mission parameters.
payload_mass    = sys.provides("payload_mass",    unit="kg")
operating_time  = sys.provides("operating_time",  unit="s")
elbow_speed     = sys.provides("elbow_speed",     unit="rad/s")
shoulder_speed  = sys.provides("shoulder_speed",  unit="rad/s")
control_rate    = sys.provides("control_rate",   unit="Hz")
elbow_arm       = sys.provides("elbow_arm",      unit="m")
shoulder_arm    = sys.provides("shoulder_arm",   unit="m")

total_mass = sys.requires("total_mass", unit="kg")

elbow      = sys.add("elbow",      Joint())
shoulder   = sys.add("shoulder",   Joint(motor_density=0.25))
sensor     = sys.add("sensor",     Sensor())
controller = sys.add("controller", Controller())
battery    = sys.add("battery",    Battery())

# Mechanical chain: the elbow lifts only the payload at its arm length,
# the shoulder lifts the payload plus the elbow joint itself at its
# longer arm.
elbow.torque    >= G * payload_mass * elbow_arm
elbow.speed     >= elbow_speed
shoulder.torque >= G * (payload_mass + elbow.mass) * shoulder_arm
shoulder.speed  >= shoulder_speed

# Electronics: sensor must Nyquist the control loop; controller must
# keep up with both incoming samples and outgoing commands.
sensor.sample_rate      >= 2.0 * control_rate
controller.input_rate   >= 2.0 * control_rate
controller.command_rate >= control_rate

# Energy budget: battery sized by integrated power over the mission.
battery.energy >= operating_time * (
    elbow.electric_power + shoulder.electric_power
    + controller.power + sensor.power
)

# Aggregate mass.
total_mass >= (
    payload_mass
    + elbow.mass + shoulder.mass
    + sensor.mass + controller.mass
    + battery.mass
)

print(sys)

System('robotic_arm'):
  provides:
    payload_mass: R+[kg]
    operating_time: R+[s]
    elbow_speed: R+[rad/s]
    shoulder_speed: R+[rad/s]
    control_rate: R+[Hz]
    elbow_arm: R+[m]
    shoulder_arm: R+[m]
  requires:
    total_mass: R+[kg]
  subsystems:
    elbow: (torque, speed) -> (mass, electric_power)
    shoulder: (torque, speed) -> (mass, electric_power)
    sensor: (sample_rate) -> (power, mass)
    controller: (input_rate, command_rate) -> (power, mass)
    battery: (energy) -> (mass)
  constraints:
    elbow.torque >= ((9.81 * payload_mass) * elbow_arm)
    elbow.speed >= elbow_speed
    shoulder.torque >= ((9.81 * (payload_mass + elbow.mass)) * shoulder_arm)
    shoulder.speed >= shoulder_speed
    sensor.sample_rate >= (2.0 * control_rate)
    controller.input_rate >= (2.0 * control_rate)
    controller.command_rate >= control_rate
    battery.energy >= (operating_time * (((elbow.electric_power + shoulder.electric_power) + controller.power) + sensor.power))
    total

## Build and solve

In [4]:
arm = sys.build()

cases = [
    ("Pick-and-place light", dict(
        payload_mass=0.5, operating_time=300.0,
        elbow_speed=2.0, shoulder_speed=1.5,
        control_rate=100.0,
        elbow_arm=0.3, shoulder_arm=0.5,
    )),
    ("Heavier payload", dict(
        payload_mass=2.0, operating_time=600.0,
        elbow_speed=1.5, shoulder_speed=1.0,
        control_rate=200.0,
        elbow_arm=0.3, shoulder_arm=0.5,
    )),
    ("Long-reach precise", dict(
        payload_mass=1.0, operating_time=900.0,
        elbow_speed=1.0, shoulder_speed=0.8,
        control_rate=500.0,
        elbow_arm=0.6, shoulder_arm=0.9,
    )),
]
for label, f in cases:
    result = solve(arm, f, max_iter=300)
    print(f"{label}:")
    print(f"   iters={result.iterations}, feasible={result.feasible}")
    if result.feasible:
        for p in result.antichain.points:
            print(f"   total_mass = {p['total_mass']:.2f} kg")
    print()

Pick-and-place light:
   iters=4, feasible=True
   total_mass = 1.97 kg

Heavier payload:
   iters=4, feasible=True
   total_mass = 7.30 kg

Long-reach precise:
   iters=4, feasible=True
   total_mass = 7.24 kg



## What the DSL bought us

The eight constraint lines above each express a single physical relationship. In the lambda-based form, the same model would have grown to roughly twenty-five lines of `lambda x: x["something.something"] + ...` boilerplate, with every port name as a string. The operator-overloaded version:

- catches typos: `elbow.toruqe` becomes an `AttributeError` on the line where the typo happens, not deep inside `build()`,
- catches semantic mistakes: trying to put a module F port (e.g. `sensor.sample_rate`) on the RHS of an expression raises immediately, with a message explaining the rule,
- prints constraints as readable inequalities in `print(sys)`,
- composes through expression trees so refactoring is just arithmetic.

The cyclic dependencies (shoulder torque depending on elbow mass; battery energy depending on both joints' powers, which depend on their torques, which depend on the elbow mass which depends on the payload) are resolved transparently by the Kleene iteration: four iterations for all three test cases.
